In [1]:
from ucimlrepo import fetch_ucirepo 
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from pandas import DataFrame
import numpy as np
from torch.nn import functional as F
import polars as pl
import mlflow
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
from torchinfo import summary

# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 

{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

In [2]:
mlflow.set_tracking_uri(uri="http://192.168.100.203:5000/")
mlflow.set_experiment("[P] Linear Regression - Wine Quality")

dataset = mlflow.data.from_pandas(
   pd.concat([X, y], axis=1), name=wine_quality.metadata["name"], targets=wine_quality.metadata.target_col[0]
)


In [3]:
X.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
count,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000
mean,7.215307,0.339666,0.318633,5.443235,0.056034,30.525319,115.744574,0.994697,3.218501,0.531268,10.491801
std,1.296434,0.164636,0.145318,4.757804,0.035034,17.749400,56.521855,0.002999,0.160787,0.148806,1.192712
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.000000,77.000000,0.992340,3.110000,0.430000,9.500000
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.000000,118.000000,0.994890,3.210000,0.510000,10.300000
75%,7.700000,0.400000,0.390000,8.100000,0.065000,41.000000,156.000000,0.996990,3.320000,0.600000,11.300000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000


In [7]:
y

,quality
0,5
1,5
2,5
3,6
4,5
...,...
6492,6
6493,5
6494,6
6495,7


In [4]:
def preprocess(X: DataFrame, y: DataFrame, encode_labels: bool = True) -> tuple:
    assert isinstance(X, DataFrame)
    assert isinstance(y, DataFrame)
    assert X.shape[1] >= 1
    assert y.shape[1] == 1

    mapping = {
        "x": {idx: col for idx, col in enumerate(X.columns)},
        "y": {pl.DataFrame(y).unique().to_torch()},
    }

    X_processed: np.ndarray = X.to_numpy()
    y_processed: np.ndarray = y.to_numpy()


    return (
        torch.tensor(X_processed, dtype=torch.float32),
        torch.tensor(y_processed, dtype=torch.float32),
        mapping,
    )

X_p, y_p, mapping = preprocess(X, y)
X_train, X_test, y_train, y_test = train_test_split(
    X_p, y_p, test_size=0.20, random_state=1
)

In [5]:
class Regressor(nn.Module):
    def __init__(self, input_size, hidden_size=16, output_size=1):
        super().__init__()

        self.fc1 = nn.Linear(input_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        return x

In [6]:
device: torch.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = Regressor(len(X.columns))

params = {
    "learning_rate": 0.01,
    "epochs": 1000,
    "optimizer": optim.Adam(model.parameters(), lr=0.1),
    "device": str(device),
    "criterion": nn.MSELoss(),
}


try:
    with mlflow.start_run(log_system_metrics=True) as run:
        mlflow.set_tag("purpose", "practice")
        mlflow.set_tag("framework", "pytorch")
        mlflow.set_tag("task", "regression")
        mlflow.log_input(dataset, context="training")
        mlflow.log_params(params)
        X_train = X_train.to(device)
        y_train = y_train.to(device)
        X_test = X_test.to(device)
        y_test = y_test.to(device)

        with open("model_summary.txt", "w") as f:
            f.write(str(summary(model)))
        mlflow.log_artifact("model_summary.txt")
        model.to(device)

        for epoch in range(params["epochs"]):
            model.train()
            params["optimizer"].zero_grad()

            outputs = model(X_train)

            loss = params["criterion"](outputs, y_train)
            loss.backward()
            params["optimizer"].step()

            model.eval()
            with torch.no_grad():
                y_pred = model(X_test)
                mse = mean_squared_error(y_test.cpu(), y_pred.cpu())
                mae = mean_absolute_error(y_test.cpu(), y_pred.cpu())
                r2 = r2_score(y_test.cpu(), y_pred.cpu())

            mlflow.log_metrics(
                {
                    "loss": loss.item(),
                    "mean squared error": mse,
                    "mean absolute error": mae,
                    "r2 score": r2,
                },
                step=epoch,
            )

        # 5. Log the trained model
        model_info = mlflow.pytorch.log_model(model.cpu(), name="Regressor", input_example=X_test.cpu().numpy()[0])
except ValueError as e:
    print(f"An error occurred: {e}")
    mlflow.end_run(status="FAILED")
finally:
    model.to("cpu")
    

2025/06/30 21:54:38 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/06/30 21:54:38 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
/home/iragca/Documents/github/capstone-project-2/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-

🏃 View run bald-cat-254 at: http://192.168.100.203:5000/#/experiments/1/runs/dd35830e21584ba8a57559f638afba29
🧪 View experiment at: http://192.168.100.203:5000/#/experiments/1
